# Step 2: Feature Engineering

## What is this?
Machine Learning models (like LightGBM) can't just "look at the chart" like humans. We need to give them numbers that describe the situation. These numbers are called **Features**.

## The Plan
1.  **Calendar Features**: Tell the model what day it is (Friday? Christmas? Super Bowl?).
2.  **Lag Features**: Tell the model what happened in the past (Sales 7 days ago, 28 days ago).
3.  **Rolling Features**: Tell the model about recent trends (Average sales of the last 7 days).

We will create a big table called `data` that has everything.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Load Data
train = pd.read_csv("../data/train.csv")
calendar = pd.read_csv("../data/calendar_events.csv")

# Convert dates
train["date"] = pd.to_datetime(train["date"])
calendar["date"] = pd.to_datetime(calendar["date"])

print("Train shape:", train.shape)
print("Calendar shape:", calendar.shape)

Train shape: (18766, 4)
Calendar shape: (162, 2)


## 1. Calendar Features
We merge the `calendar_events.csv` to get holiday info. We also extract basic date parts.

In [4]:
# Merge calendar events
# We use 'left' join because we want to keep all training rows
data = train.merge(calendar, on="date", how="left")

# Fill missing events with "None"
data["event"] = data["event"].fillna("NoEvent")

# Extract basic date features
data["day_of_week"] = data["date"].dt.dayofweek  # 0=Monday, 6=Sunday
data["month"] = data["date"].dt.month
data["year"] = data["date"].dt.year
data["day_of_month"] = data["date"].dt.day

# Is it a weekend?
data["is_weekend"] = (data["day_of_week"] >= 5).astype(int)

data[["date", "event", "day_of_week", "is_weekend"]].head()

,date,event,day_of_week,is_weekend
0,2011-01-29,NoEvent,5,1
1,2011-01-30,NoEvent,6,1
2,2011-01-31,NoEvent,0,0
3,2011-02-01,NoEvent,1,0
4,2011-02-02,NoEvent,2,0


## 2. Lag Features
"Lag" means looking back in time. 
- `lag_28`: Sales 28 days ago. (Why 28? Because in the competition we predict 28 days ahead, so we always have data from 28 days ago available!)
- `lag_35`: Sales 35 days ago (5 weeks ago).
- `lag_365`: Sales 1 year ago.

In [5]:
# Sort by store and date to ensure shifts work correctly
data = data.sort_values(["store_id", "date"])

# Create Lags
lags = [28, 35, 42, 49, 56, 365]

for lag in lags:
    data[f"lag_{lag}"] = data.groupby("store_id")["revenue"].shift(lag)

# Check if it worked
data[["date", "store_id", "revenue", "lag_28"]].tail()

,date,store_id,revenue,lag_28
18761,2015-09-26,10,25689.55,23304.33
18762,2015-09-27,10,26557.53,24047.72
18763,2015-09-28,10,19067.53,20356.75
18764,2015-09-29,10,16467.95,21687.47
18765,2015-09-30,10,17452.25,22361.53


## 3. Rolling Features
"Rolling" means taking an average over a window.
- `rolling_mean_7`: Average sales of the last 7 days (shifted by 28 days to be safe).
- `rolling_mean_28`: Average sales of the last 28 days.

In [6]:
# We shift by 28 days FIRST, then calculate the rolling mean.
# This ensures we don't use data we wouldn't have at prediction time.
data["rolling_mean_7"] = data.groupby("store_id")["revenue"].transform(lambda x: x.shift(28).rolling(7).mean())
data["rolling_mean_28"] = data.groupby("store_id")["revenue"].transform(lambda x: x.shift(28).rolling(28).mean())

data[["date", "store_id", "revenue", "rolling_mean_7"]].tail()

,date,store_id,revenue,rolling_mean_7
18761,2015-09-26,10,25689.55,19489.088571
18762,2015-09-27,10,26557.53,19682.270000
18763,2015-09-28,10,19067.53,20007.281429
18764,2015-09-29,10,16467.95,20556.612857
18765,2015-09-30,10,17452.25,21363.764286


## Save for Modeling
We drop rows with NaN values (the first few months won't have lags) and save.

In [7]:
# Drop rows where we don't have enough history for lags (approx first year)
data = data.dropna()

print("Final shape for training:", data.shape)

# Save to a new CSV (parquet is faster, but let's stick to CSV for simplicity if it fits)
data.to_csv("../data/train_features.csv", index=False)
print("Saved ../data/train_features.csv")

Final shape for training: (14751, 18)
Saved ../data/train_features.csv
